# EOF / PCA of a brightness-temperature sequence

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract the loop

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()
N = 6

fields = session.extract_fields('''
adde = dict(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
            size='ALL', unit='TEMP', mag=(-8, -8))
frames = [loadADDEImage(position=p, **adde) for p in range(-(N-1), 1)]
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Sequence Display', frames)
''', values={'N': N}, times=range(N))

cube = np.stack([f.masked() for f in fields])
nav = fields[0]
print(cube.shape, nav.unit)

## 2. EOFs of the temperature anomalies

In [ ]:
from sklearn.decomposition import PCA

valid = np.all(np.isfinite(cube), axis=0) & nav.valid
M = cube[:, valid]
M = M - M.mean(0, keepdims=True)

pca = PCA(n_components=4).fit(M)
pcs = pca.transform(M)
print('explained variance:', np.round(pca.explained_variance_ratio_, 3))
print('anomaly range: %.1f .. %.1f K' % (M.min(), M.max()))

def to_map(vec):
    out = np.full(cube.shape[1:], np.nan); out[valid] = vec; return out
eofs = [to_map(pca.components_[k]) for k in range(4)]

## 3. Plot EOFs and PC time series

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(13, 6))
for k in range(4):
    a = ax.flat[k]; vmax = np.nanmax(np.abs(eofs[k]))
    a.imshow(eofs[k], cmap='RdBu_r', vmin=-vmax, vmax=vmax); a.axis('off')
    a.set_title('EOF %d (%.0f%%)' % (k + 1, 100 * pca.explained_variance_ratio_[k]))
ax.flat[4].plot(pcs[:, :3], '-o'); ax.flat[4].set_title('PC time series')
ax.flat[4].set_xlabel('frame'); ax.flat[5].axis('off')
plt.tight_layout()

## 4. Display EOF 1 in McIDAS-V

In [ ]:
from scipy.interpolate import griddata

def regrid(field, values, nlat=180, nlon=300, fill=np.nan):
    m = field.valid & np.isfinite(values)
    pts = np.column_stack([field.lats[m], field.lons[m]])
    glats = np.linspace(np.nanmax(field.lats[m]), np.nanmin(field.lats[m]), nlat)
    glons = np.linspace(np.nanmin(field.lons[m]), np.nanmax(field.lons[m]), nlon)
    GLA, GLO = np.meshgrid(glats, glons, indexing='ij')
    g = griddata(pts, values[m], (GLA, GLO), method='linear')
    return np.where(np.isfinite(g), g, fill).astype('f4'), glats, glons

grid, glats, glons = regrid(nav, eofs[0], fill=0.0)
session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setLayerLabel(label='EOF 1 of Tb variability')
''', arrays={'g': (grid, glats, glons)})